# Lab01 - sprint 2
## Подготовка: предсказание quality по alcohol

[Описание](#description) · [Данные](#data) · [Разделение](#split) · [Проверки](#tests) · [Выводы](#resume)

<a id="description"></a>
### Description
Вход модели — только `alcohol` (% об.). Цель — `quality`, порядковая оценка дегустаторов.
Регрессия будет приближать среднюю числовую оценку при данном содержании спирта;
это допущение о шкале, а не доказательство причинного влияния алкоголя.
`wine_type` используется для контроля состава частей, но не входит в X.

Этот этап готовит независимый от обучения holdout. Затем сравним константный прогноз,
линейную регрессию и kNN. Подбор моделей и преобразований — только внутри train.
Генератор условного распределения качества — отдельный следующий этап:
одна регрессия не выполняет требование лабораторной о ГСЧ.

In [1]:
from pathlib import Path
import sys
import hashlib
import json
import platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'src'))
from wine_stats.data import load_wines
from wine_stats.split import split_wines, model_parts

SEED = 42
TEST_FRACTION = 0.2
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__})

{'python': '3.12.14', 'numpy': '2.3.5', 'pandas': '2.2.3'}


<a id="data"></a>
### Данные и аудит
Источник: [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality).
Исходный [ZIP](https://archive.ics.uci.edu/static/public/186/wine+quality.zip).
Поместите оба CSV в `data/raw/` (инструкция: `data/README.md`).
CSV не изменяются; повторы сохраняются. Причина их появления неизвестна.
Контрольные суммы фиксируют точные входные файлы.

In [2]:
raw = ROOT / 'data/raw'
df = load_wines(raw)
source_info = {
    f'winequality-{kind}.csv': {
        'sha256': hashlib.sha256((raw / f'winequality-{kind}.csv').read_bytes()).hexdigest(),
        'rows': int(df.wine_type.eq(kind).sum())
    } for kind in ('red', 'white')
}
audit = pd.DataFrame([{
    'rows': len(df), 'columns': len(df.columns),
    'missing_cells': int(df.isna().sum().sum()),
    'full_duplicate_rows': int(df.duplicated().sum()),
    'unique_rows': len(df.drop_duplicates())
}])
display(audit)
display(df.dtypes.rename('dtype').to_frame())
print(json.dumps(source_info, indent=2))

{
  "winequality-red.csv": {
    "sha256": "4a402cf041b025d4566d954c3b9ba8635a3a8a01e039005d97d6a710278cf05e",
    "rows": 1599
  },
  "winequality-white.csv": {
    "sha256": "76c3f809815c17c07212622f776311faeb31e87610d52c26d87d6e361b169836",
    "rows": 4898
  }
}


,rows,columns,missing_cells,full_duplicate_rows,unique_rows
0,6497,13,0,1177,5320


,dtype
fixed_acidity,float64
volatile_acidity,float64
citric_acid,float64
residual_sugar,float64
chlorides,float64
free_sulfur_dioxide,float64
total_sulfur_dioxide,float64
density,float64
pH,float64
sulphates,float64


<a id="split"></a>
### Разделение без пересечения полных повторов
Одинаковые строки по всем 11 измерениям, качеству и типу вина образуют одну группу.
Совпадение только пары `alcohol, quality` не означает дубликат.
В каждом типе вина случайно выбираем около 20% **групп** для test, остальные — train.
Поэтому доля строк может немного отличаться от 20%.
SHA-256 содержимого задаёт идентификатор группы; `red:1` — первая строка данных красного CSV.

Группировка предотвращает утечку известных полных повторов, но не гарантирует независимость
неизвестных партий/производителей: их идентификаторов в данных нет.
Seed фиксирован заранее; распределение quality не используется для выбора удачного разбиения.

In [3]:
manifest = split_wines(df, test_fraction=TEST_FRACTION, seed=SEED)
parts = model_parts(df, manifest)
X_train, y_train = parts['train']
X_test, y_test = parts['test']
summary = (manifest.groupby(['split', 'wine_type'])
           .agg(rows=('row_id', 'size'), groups=('group_id', 'nunique'))
           .reset_index())
summary['share_within_type'] = summary['rows'] / summary.wine_type.map(df.wine_type.value_counts())
display(summary)
print({'X_train': X_train.shape, 'X_test': X_test.shape})
# Здесь нет масштабирования и обучения; test сохраняется для финальной оценки.

{'X_train': (5195, 1), 'X_test': (1302, 1)}


,split,wine_type,rows,groups,share_within_type
0,test,red,314,272,0.196373
1,test,white,988,792,0.201715
2,train,red,1285,1087,0.803627
3,train,white,3910,3169,0.798285


<a id="tests"></a>
### Tests и сохранение разбиения

In [4]:
assert len(manifest) == len(df) and manifest.row_id.is_unique
assert manifest.groupby('group_id').split.nunique().max() == 1
assert set(manifest.split) == {'train', 'test'}
assert len(X_train) + len(X_test) == len(df)
assert list(X_train.columns) == list(X_test.columns) == ['alcohol']
assert set(X_train.index).isdisjoint(X_test.index)
pd.testing.assert_frame_equal(manifest, split_wines(df, seed=SEED, test_fraction=TEST_FRACTION))

out = ROOT / 'data/processed'
out.mkdir(parents=True, exist_ok=True)
manifest.to_csv(out / 'alcohol_quality_split.csv', index=False)
metadata = {
    'seed': SEED, 'test_group_fraction': TEST_FRACTION,
    'features': ['alcohol'], 'target': 'quality',
    'source': 'https://archive.ics.uci.edu/static/public/186/wine+quality.zip',
    'files': source_info,
    'manifest_sha256': hashlib.sha256((out / 'alcohol_quality_split.csv').read_bytes()).hexdigest(),
    'numpy': np.__version__, 'pandas': pd.__version__,
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
(out / 'alcohol_quality_split.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
summary.to_csv(out / 'alcohol_quality_split_summary.csv', index=False)
print('Проверки пройдены; manifest, metadata и summary сохранены в data/processed.')

Проверки пройдены; manifest, metadata и summary сохранены в data/processed.


<a id="resume"></a>
## Lab resume
Исходные данные: 6497 наблюдений, 1177 полных повторов сверх первых вхождений,
пропусков нет. Повторы сохранены; группы не пересекают границу train/test.
При seed=42 получено 5195 обучающих и 1302 тестовых наблюдения
(20.04% строк в test). В test 314 красных и 988 белых вин.
Доля test внутри типов: 19.64% и 20.17% соответственно.

Подготовлены X с одним признаком `alcohol` и y=`quality`.
Модель ещё не обучена; вывод о предсказательной силе спирта пока не сделан.
Следующий этап — baseline и линейная регрессия с оценкой MAE, RMSE и R²;
для выбора модели потребуется валидация внутри train.

### cat_mark_table
Официальные cat_id и coeff пока не предоставлены; не присваиваем их самостоятельно.

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| — | [Подготовка данных](00_data_split.ipynb#split) | — | Выполнены аудит, воспроизводимое разделение и проверки; это подготовка, не готовый ГСЧ. |

Проверка выполнения: все кодовые ячейки последовательно исполнены через IPython
в чистом процессе; запуск отдельного Jupyter-ядра в проверочной среде недоступен из-за ограничения сокетов.